In [1]:
# import libraries

import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3



# 1.IMPORTING DATA 

In [2]:
# first of all take the name of excel file
excel_file_s="customer_churn_practice_expanded.xlsx"

# then create  connection python to sqlite3 by openinig database
connection=sqlite3.connect("customer_churn.db")

# now read excel file 
excel_data = pd.ExcelFile(excel_file_s)

#now read all the sheet  one by one and store it in data frame
for i in excel_data.sheet_names:

    df=pd.read_excel(excel_data,sheet_name=i) # read sheet in the data frame

    # now these sheet sote in the sqllite in the form of table
    df.to_sql(name=i,     # table name = sheet name
              con=connection,
              if_exists='replace',
              index=False
              )

connection.close()    

print("Now all the excel file conerted into sqllite  table Successfully")

Now all the excel file conerted into sqllite  table Successfully


In [3]:
# Data Import: db file to pandas, storing each table to a separate df

# Connect to SQLite database
conn = sqlite3.connect('customer_churn.db')

# sql query to Get all table names
sql_query = """
        SELECT name
        FROM sqlite_master
        WHERE type='table';
"""

# read sql query in pandas
tables = pd.read_sql(sql_query, conn)

# create dataframe for each table
for table_name in tables['name']:
    df = pd.read_sql(f"SELECT * FROM {table_name}", conn) # Read table into dataframe
    globals()[f"df_{table_name}"] = df                    # Create dynamic dataframe name
    print(f"Created dataframe: df_{table_name}")

# Close connection
conn.close()

Created dataframe: df_db_customer
Created dataframe: df_db_subscription
Created dataframe: df_db_support


In [4]:
# Print table names and column names
conn = sqlite3.connect('customer_churn.db')

for table_name in tables['name']:
    print(f"\nTable Name: {table_name}")
    # Get column information
    columns_query = f"PRAGMA table_info({table_name});"
    columns = pd.read_sql(columns_query, conn)
    print("Columns:")
    print(columns['name'].tolist())

# Close connection
conn.close()


Table Name: db_customer
Columns:
['customerid', 'name', 'country', 'state', 'gender', 'dob', 'interests', 'pincode']

Table Name: db_subscription
Columns:
['customerid', 'subscription_start_date', 'subscription_type', 'renewal_date', 'plan_type', 'contract_type', 'cancellation_date', 'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score']

Table Name: db_support
Columns:
['customerid', 'complaint_date', 'escalations', 'csat_score', 'col_1', 'comment']


# 2.DATA CLEANING

## A. df_db_customer

In [6]:
df_db_customer.head()

,customerid,name,country,state,gender,dob,interests,pincode
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12 00:00:00,travel,NaN
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23 00:00:00,NaN,NaN
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15 00:00:00,movie,NaN
3,0011-IGKFF,mohan,India,Nagaland,Male,2001-08-30 00:00:00,NaN,NaN
4,0013-EXCHZ,mira,India,Delhi,Female,1990-05-05 00:00:00,drama,NaN


In [7]:
df_db_customer.tail()

,customerid,name,country,state,gender,dob,interests,pincode
997,CUST-00977,Simran Gupta,Nepal,Haryana,Male,1975-11-14 19:47:27,NaN,672601.0
998,CUST-00978,Karan Rao,Nepal,Haryana,Female,1967-03-10 11:04:29,technology,604626.0
999,CUST-00979,Isha Khan,India,Gujarat,Male,1997-11-07 10:45:00,music,289223.0
1000,CUST-00005,Nisha Joshi,India,Tamil Nadu,Female,2002-06-23 17:09:32,fitness,672601.0
1001,CUST-00060,Kabir Khan,India,West Bengal,Male,1998-05-09 23:09:53,sports,122221.0


In [8]:
df_db_customer.info()

<class 'pandas.DataFrame'>
RangeIndex: 1002 entries, 0 to 1001
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   customerid  1002 non-null   str    
 1   name        1002 non-null   str    
 2   country     987 non-null    str    
 3   state       1002 non-null   str    
 4   gender      992 non-null    str    
 5   dob         1002 non-null   str    
 6   interests   894 non-null    str    
 7   pincode     981 non-null    float64
dtypes: float64(1), str(7)
memory usage: 62.8 KB


In [9]:
df_db_customer.isnull().sum()

customerid      0
name            0
country        15
state           0
gender         10
dob             0
interests     108
pincode        21
dtype: int64

In [10]:
df_db_customer["gender"].unique()

<StringArray>
['Male', 'Female', 'Women', 'Men', nan]
Length: 5, dtype: str

In [11]:
df_db_customer["country"].isnull().head(50)

0     False
1     False
2     False
3     False
4     False
5      True
6     False
7     False
8      True
9     False
10    False
11    False
12     True
13    False
14    False
15    False
16    False
17    False
18    False
19    False
20    False
21    False
22    False
23    False
24    False
25    False
26    False
27    False
28    False
29    False
30    False
31    False
32    False
33    False
34    False
35    False
36    False
37    False
38    False
39    False
40    False
41    False
42    False
43    False
44    False
45    False
46    False
47    False
48    False
49    False
Name: country, dtype: bool

In [12]:
df_db_customer[["name","country"]].iloc[0:40]

,name,country
0,keshav,India
1,raghav,India
2,lalita,India
3,mohan,India
4,mira,India
5,durga,NaN
6,mina,India
7,madan,India
8,maya,NaN
9,arjun,Nepal


In [13]:
# so now what to do 
""" 
1.drop interest and pincode columns(becuz we don't need and many null value present)
2.rename name to customer_name (for better understanding)
3.standarized (in gender set only two  unique value i.e (male,female))
4.change data type of dob
5.set country as thier name 

"""

" \n1.drop interest and pincode columns(becuz we don't need and many null value present)\n2.rename name to customer_name (for better understanding)\n3.standarized (in gender set only two  unique value i.e (male,female))\n4.change data type of dob\n5.set country as thier name \n\n"

### A.drop interest and pindcode column

In [14]:

df_db_customer.drop(columns=['interests','pincode'],inplace=True)

In [15]:
df_db_customer.head()

,customerid,name,country,state,gender,dob
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12 00:00:00
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23 00:00:00
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15 00:00:00
3,0011-IGKFF,mohan,India,Nagaland,Male,2001-08-30 00:00:00
4,0013-EXCHZ,mira,India,Delhi,Female,1990-05-05 00:00:00


 ### b.rename column ---name to customer_name

In [16]:

df_db_customer.rename(columns={"name" : "customer_name"}, inplace= True)

In [17]:
df_db_customer.head()

,customerid,customer_name,country,state,gender,dob
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12 00:00:00
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23 00:00:00
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15 00:00:00
3,0011-IGKFF,mohan,India,Nagaland,Male,2001-08-30 00:00:00
4,0013-EXCHZ,mira,India,Delhi,Female,1990-05-05 00:00:00


### c. standardize (gender column fixing)

In [18]:
df_db_customer['gender']=df_db_customer['gender'].replace({"Men":"Male", "Women":"Female"})

In [19]:
df_db_customer['gender'].unique()

<StringArray>
['Male', 'Female', nan]
Length: 3, dtype: str

### d.change datatype

In [20]:
df_db_customer['dob']=pd.to_datetime(df_db_customer['dob'])

In [21]:
df_db_customer.head()

,customerid,customer_name,country,state,gender,dob
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15
3,0011-IGKFF,mohan,India,Nagaland,Male,2001-08-30
4,0013-EXCHZ,mira,India,Delhi,Female,1990-05-05


### e. Fix missing values

In [ ]:
# first off all check where  null value are present in country column 
df_db_customer[df_db_customer['country'].isna()]

,customerid,customer_name,country,state,gender,dob
5,0013-MHZWF,durga,NaN,Delhi,Female,1988-12-10 00:00:00
8,0015-UOCOJ,maya,NaN,Kathmandu,Female,1985-07-07 00:00:00
12,0018-NYROU,chitra,NaN,Telangana,Female,2004-12-01 00:00:00
87,CUST-00067,Aarav Singh,NaN,Tamil Nadu,Male,2004-03-13 22:38:59
245,CUST-00225,Vivaan Das,NaN,Maharashtra,Male,2000-06-02 15:27:48
657,CUST-00637,Simran Mehta,NaN,Bihar,Female,1990-07-23 06:19:45
665,CUST-00645,Simran Singh,NaN,Bihar,Female,1991-02-18 17:37:05
685,CUST-00665,Isha Das,NaN,Madhya Pradesh,Male,1980-04-20 00:45:17
718,CUST-00698,Nisha Rao,NaN,Maharashtra,Male,1979-04-23 07:59:50
744,CUST-00724,Rohan Singh,NaN,Bihar,Male,1973-03-25 20:16:12


In [ ]:
# now then check total no. of  null value presnet in country column  
df_db_customer['country'].isna().sum()

np.int64(15)

In [25]:
# drop the column where country is null and create a dictionary in which index set [state] and column is [country] 
mapping_country_state=df_db_customer.dropna(subset=['country']).set_index('state')['country'].to_dict()

In [ ]:
# then now fill the country name in the country column on the basis of state by using map--the map can be search their state name of coutry by taking help which i create dictinory of statecountry
df_db_customer['country']=df_db_customer['country'].fillna(df_db_customer['state'].map(mapping_country_state))

In [28]:
# now check any null values are present or not ? 
df_db_customer['country'].isna().sum()
df_db_customer[df_db_customer['country'].isna()]

,customerid,customer_name,country,state,gender,dob


##### now df_db_customer  table is clean

## B. df_db_subcription 

In [29]:
df_db_subscription.info()

<class 'pandas.DataFrame'>
RangeIndex: 1001 entries, 0 to 1000
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   customerid               1001 non-null   str  
 1   subscription_start_date  1001 non-null   str  
 2   subscription_type        1001 non-null   str  
 3   renewal_date             1001 non-null   str  
 4   plan_type                1001 non-null   str  
 5   contract_type            1001 non-null   str  
 6   cancellation_date        347 non-null    str  
 7   cancellation_reason      347 non-null    str  
 8   monthly_charges          996 non-null    str  
 9   cltv                     1001 non-null   int64
 10  churn_score              996 non-null    str  
dtypes: int64(1), str(10)
memory usage: 86.2 KB


In [30]:
df_db_subscription[df_db_subscription.isna()]

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
997,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
998,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_db_subscription.isna().sum()
# here cancellation date  and cancellation reasion is null it means the user  use ur webite from starting to end 

customerid                   0
subscription_start_date      0
subscription_type            0
renewal_date                 0
plan_type                    0
contract_type                0
cancellation_date          654
cancellation_reason        654
monthly_charges              5
cltv                         0
churn_score                  5
dtype: int64

In [32]:
df_db_subscription.head()

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,NaN,NaN,13.99,627,12
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,91
2,0004-TLHLJ,2022-11-20,Organic,2025-11-20,Basic,Monthly,NaN,NaN,6.99,210,34
3,0011-IGKFF,2019-05-10,Paid,2025-05-10,Premium,Annual,NaN,NaN,22.99,1725,8
4,0013-EXCHZ,2023-01-05,Refferal,2024-01-05,Standard,Monthly,2024-02-28,Too expensive,13.99,195,88


In [33]:
df_db_subscription.tail()

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score
996,CUST-00976,2026-02-22 00:00:00,Paid,2026-11-21 00:00:00,Basic,Monthly,NaN,NaN,60.37,615,56
997,CUST-00977,2022-05-22 00:00:00,Paid,2023-01-14 00:00:00,Standard,Monthly,2026-07-29 00:00:00,Poor streaming quality,80.65,1078,99
998,CUST-00978,2023-08-02 00:00:00,Paid,2024-03-08 00:00:00,Premium,Monthly,NaN,NaN,22.36,602,40
999,CUST-00979,2024-04-22 00:00:00,Paid,2025-01-31 00:00:00,Premium,Annual,NaN,NaN,23.27,586,25
1000,CUST-00010,2022-08-21 00:00:00,Referral,2023-06-22 00:00:00,Standard,Annual,2022-11-23 00:00:00,Switched to competitor,81.59,775,39


In [ ]:
# so now fix datatype of date 
df_db_subscription['subscription_start_date']=pd.to_datetime(df_db_subscription['subscription_start_date'],format='mixed')
df_db_subscription['renewal_date']=pd.to_datetime(df_db_subscription['renewal_date'],format='mixed')
df_db_subscription['cancellation_date']=pd.to_datetime(df_db_subscription['cancellation_date'],format='mixed')

In [42]:
df_db_subscription.tail()

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score
996,CUST-00976,2026-02-22,Paid,2026-11-21,Basic,Monthly,NaT,NaN,60.37,615,56
997,CUST-00977,2022-05-22,Paid,2023-01-14,Standard,Monthly,2026-07-29,Poor streaming quality,80.65,1078,99
998,CUST-00978,2023-08-02,Paid,2024-03-08,Premium,Monthly,NaT,NaN,22.36,602,40
999,CUST-00979,2024-04-22,Paid,2025-01-31,Premium,Annual,NaT,NaN,23.27,586,25
1000,CUST-00010,2022-08-21,Referral,2023-06-22,Standard,Annual,2022-11-23,Switched to competitor,81.59,775,39


## C.  df_db_support

In [45]:
df_db_support.info()


<class 'pandas.DataFrame'>
RangeIndex: 707 entries, 0 to 706
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   customerid      707 non-null    str   
 1   complaint_date  707 non-null    str   
 2   escalations     707 non-null    str   
 3   csat_score      702 non-null    str   
 4   col_1           0 non-null      object
 5   comment         562 non-null    str   
dtypes: object(1), str(5)
memory usage: 33.3+ KB


In [46]:
df_db_support.head()

,customerid,complaint_date,escalations,csat_score,col_1,comment
0,0003-MKNFE,2024-08-28 00:00:00,N,60,None,service issue
1,0003-MKNFE,2024-08-28 00:00:00,Y,10,None,demaned refund
2,0013-EXCHZ,2024-01-20 00:00:00,Y,20,None,NaN
3,0013-MHZWF,2025-03-18 00:00:00,N,90,None,guidance to renew
4,0013-SMEOE,2024-11-01 00:00:00,N,30,None,NaN


In [62]:
""" 
here only two thing is for doing 
1 remove column  col_1 and comment
2 fix datatype of complaint_date
"""

' \nhere only two thing is for doing \n1 remove column  col_1 and comment\n2 fix datatype of complaint_date\n'

In [52]:
df_db_support.drop(columns=['col_1','comment'] ,inplace=True)

In [60]:
df_db_support["complaint_date"]=pd.to_datetime(df_db_support["complaint_date"])

In [61]:
df_db_support.head()

,customerid,complaint_date,escalations,csat_score
0,0003-MKNFE,2024-08-28,N,60
1,0003-MKNFE,2024-08-28,Y,10
2,0013-EXCHZ,2024-01-20,Y,20
3,0013-MHZWF,2025-03-18,N,90
4,0013-SMEOE,2024-11-01,N,30
